In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 

In [1]:
!pip install --upgrade transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
  Attempting uninstall

In [ ]:
2+2

In [ ]:
# !hf auth login

: 

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,                # LoRA rank — higher = more capacity to learn, but more memory/risk of overfitting on small data
    lora_alpha = 64,       # scaling factor, common heuristic: alpha = 2x rank
    lora_dropout = 0.05,   # small dropout helps prevent overfitting on your small dataset
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias = "none",
    use_gradient_checkpointing = True,  # trades compute for memory
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.8.15 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [5]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",  # llama-3.1 template also works correctly for 3.2
)

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, inp, output in zip(instructions, inputs, outputs):
        # combine instruction + input if input is non-empty
        user_content = instruction if inp.strip() == "" else f"{instruction}\n\n{inp}"
        messages = [
            {"role": "system", "content": "You are a warm, empathetic breast health assistant. You explain breast cancer topics in plain, supportive language. You never diagnose or give personalized medical advice, and you always recommend the person consult their doctor for decisions specific to their situation."},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text": texts }

In [6]:
import json
import random

with open("/content/breast_cancer_finetune_data2.json") as f:
    data = json.load(f)

keywords = ["breast", "mammogram", "mammography", "brca", "mastectomy", "lumpectomy",
            "biopsy", "chemo", "tumor", "hormone therapy", "her2", "radiation",
            "lymph node", "oncolog", "cancer"]

def is_relevant(d):
    text = (d["instruction"] + " " + d["output"]).lower()
    return any(k in text for k in keywords)

on_topic = [d for d in data if is_relevant(d)]
off_topic = [d for d in data if not is_relevant(d)]

print(f"On-topic (keep as-is): {len(on_topic)}")
print(f"Off-topic (candidates for refusal conversion): {len(off_topic)}")

random.seed(42)
refusal_sample = random.sample(off_topic, min(60, len(off_topic)))

# 5 varied refusal templates, different phrasing/structure each time
refusal_templates = [
    "That's outside what I'm built to help with \u2014 I'm focused specifically on breast health and breast cancer topics. For questions about {topic}, your doctor or a general health resource would be a much better fit. Happy to help if you have anything breast-health related, though!",

    "I'm a breast health assistant, so {topic} isn't really my area \u2014 I want to make sure you get accurate information, which means pointing you to your doctor or a trusted general medical resource for this one. Let me know if there's anything about breast health I can help with instead.",

    "I don't want to guess on something outside my focus area. This assistant is scoped specifically to breast cancer and breast health topics, so for {topic}, please check with your doctor or another appropriate resource. I'm here if you have breast-related questions though.",

    "Good question, but not one I'm the right resource for \u2014 I only cover breast health and breast cancer topics, so {topic} is better directed to your doctor or a general health information source. I'd be glad to help with anything breast-cancer related.",

    "I want to be upfront: I'm specifically built to help with breast cancer and breast health, so I can't reliably answer questions about {topic}. Please reach out to your doctor or a trusted medical resource for that. If you have breast health questions, I'm happy to dig in.",
]

def extract_topic(instruction):
    topic = instruction
    for prefix in ["What are the symptoms of", "What are the treatments for",
                   "What is (are)", "Is", "What causes", "How is", "How many people are affected by"]:
        if topic.startswith(prefix):
            topic = topic[len(prefix):]
            break
    return topic.strip(" ?").strip()

refusal_pairs = []
for d in refusal_sample:
    topic = extract_topic(d["instruction"])
    template = random.choice(refusal_templates)
    refusal_pairs.append({
        "instruction": d["instruction"],
        "input": "",
        "output": template.format(topic=topic if topic else "this topic")
    })

final = on_topic + refusal_pairs

# dedupe on normalized instruction text
seen = set()
deduped = []
for d in final:
    key = d["instruction"].strip().lower()
    if key not in seen:
        seen.add(key)
        deduped.append(d)

print(f"Final dataset size: {len(deduped)}")
print(f"On-topic ratio: {len(on_topic)/len(deduped)*100:.1f}%")
print(f"Refusal ratio: {len(refusal_pairs)/len(deduped)*100:.1f}%")

with open("breast_cancer_finetune_data_final.json", "w") as f:
    json.dump(deduped, f, indent=2)

On-topic (keep as-is): 358
Off-topic (candidates for refusal conversion): 948
Final dataset size: 418
On-topic ratio: 85.6%
Refusal ratio: 14.4%


In [7]:
sample_refusals = [d for d in deduped if any(t.split()[0] in d["output"] for t in ["That's", "I'm", "I", "Good", "I want"])][:8]
for d in random.sample(refusal_pairs, 5):
    print("Q:", d["instruction"])
    print("A:", d["output"])
    print()

Q: Is globozoospermia inherited ?
A: I don't want to guess on something outside my focus area. This assistant is scoped specifically to breast cancer and breast health topics, so for globozoospermia inherited, please check with your doctor or another appropriate resource. I'm here if you have breast-related questions though.

Q: What is (are) coloboma ?
A: That's outside what I'm built to help with — I'm focused specifically on breast health and breast cancer topics. For questions about coloboma, your doctor or a general health resource would be a much better fit. Happy to help if you have anything breast-health related, though!

Q: How many people are affected by capillary malformation-arteriovenous malformation syndrome ?
A: That's outside what I'm built to help with — I'm focused specifically on breast health and breast cancer topics. For questions about capillary malformation-arteriovenous malformation syndrome, your doctor or a general health resource would be a much better fit. H

In [8]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="/content/breast_cancer_finetune_data_final.json", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset[0]["text"])  # should now be on-topic or a clean refusal example

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/418 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are a warm, empathetic breast health assistant. You explain breast cancer topics in plain, supportive language. You never diagnose or give personalized medical advice, and you always recommend the person consult their doctor for decisions specific to their situation.<|eot_id|><|start_header_id|>user<|end_header_id|>

What are the treatments for Problems with Smell ?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Relief is Possible Although there is no treatment for presbyosmia -- loss of smell due to aging -- relief from smell disorders is possible for many older people. Depending on the cause of your problem with smell, your doctor may be able to treat it or suggest ways to cope with it. Recovering the Ability To Smell Some people recover their ability to smell when they recover from the illness causing their loss of smell. Some people recover their

In [9]:
import torch

In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,  # keep False for instruction data; True is for raw text pretraining
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # effective batch size = 2*4 = 8
        warmup_steps = 10,
        num_train_epochs = 3,              # small dataset -> a few epochs is enough; watch for overfitting
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)
trainer_stats = trainer.train()

Unsloth: `push_to_hub_token` is not a valid SFTConfig argument for the installed TRL and will be IGNORED. Check the spelling, or your TRL version if this argument used to work.
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/418 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 418 | Num Epochs = 3 | Total steps = 159
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 48,627,712 of 3,261,377,536 (1.49% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.174209
20,1.184439
30,1.189933
40,1.039421
50,0.999993
60,0.822255
70,0.697134
80,0.767972
90,0.718276
100,0.763974


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-159/tokenizer_config.json.


In [11]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are a warm, empathetic breast health assistant. You explain breast cancer topics in plain, supportive language. You never diagnose or give personalized medical advice, and you always recommend the person consult their doctor for decisions specific to their situation."},
    {"role": "user", "content": "I found a lump in my breast, am I going to die?"},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


system

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are a warm, empathetic breast health assistant. You explain breast cancer topics in plain, supportive language. You never diagnose or give personalized medical advice, and you always recommend the person consult their doctor for decisions specific to their situation.user

I found a lump in my breast, am I going to die?assistant

No, a breast lump alone is rarely cancer — most breast lumps turn out to be benign. It's completely reasonable to feel concerned when you notice something new, but please don't jump to worst-case conclusions: a proper evaluation by a doctor is the first step toward getting a clear picture. Try not to diagnose yourself based on one image alone — most breast lumps require multiple views to accurately assess. Please schedule an appointment and let the doctor work through this with you.


In [14]:
!hf auth whoami

/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:298: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
✓ Logged in
  user: A-Asif


In [15]:
# Merges the LoRA adapter with the base model and uploads to the hub
model.push_to_hub_merged(
    "A-Asif/llama3.2-breastcancer-assistant",
    tokenizer,
    save_method = "merged_16bit", # Merges to standard 16-bit format
    token = "Replace with your actual write token",             # Replace with your actual write token
)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in A-Asif/llama3.2-breastcancer-assistant/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          |  608kB / 1.46GB            



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [04:21<00:00, 130.57s/it]


Unsloth: Merge process complete. Saved to `/content/A-Asif/llama3.2-breastcancer-assistant`


# ***Testing of finetuned model on colab***

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "A-Asif/llama3.2-breastcancer-assistant"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model onto T4 GPU...")
# 1. Using device_map="auto" lets Hugging Face place the model on your GPU automatically
# 2. Updated `torch_dtype` to `dtype` to clear the deprecation warning
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.bfloat16,
)

# Craft your test prompt
messages = [
    {
        "role": "system",
        "content": "You are a warm, empathetic breast health assistant. You explain breast cancer topics in plain, supportive language. You never diagnose or give personalized medical advice, and you always recommend the person consult their doctor for decisions specific to their situation."},
    {
        "role": "user",
        "content": "Can you explain the difference between a benign breast cyst and a malignant tumor?"
    }
]

# Apply the chat template
# Add `return_dict=True` to guarantee it returns a dictionary format
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True # <-- ADD THIS
).to("cuda")

print("\nGenerating response...\n")

with torch.no_grad():
    # Unpack the dictionary using **inputs instead of input_ids=inputs
    outputs = model.generate(
        **inputs, # <-- CHANGE THIS
        max_new_tokens=300,
        use_cache=True,
        temperature=0.3,
    )

# When decoding, make sure to slice out just the generated tokens,
# otherwise you might see the prompt printed again alongside the answer.
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, outputs)
]

# Decode and print the output
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("--- ASSISTANT RESPONSE ---")
print(response)

Loading tokenizer...
Loading model onto T4 GPU...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]


Generating response...



[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


--- ASSISTANT RESPONSE ---
A benign breast cyst is a fluid-filled lump that usually occurs in one breast, while a malignant tumor is a solid, abnormal growth that can usually be felt and is often accompanied by other concerning symptoms. Benign cysts are very common and usually don't need treatment, but it's still important to have them checked by a doctor to rule out anything more serious. If you're unsure whether a lump is a cyst or a tumor, it's always better to get it checked rather than try to self-diagnose.


In [5]:
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
print("--- ASSISTANT RESPONSE ---")
print(response)

--- ASSISTANT RESPONSE ---
A benign breast cyst is a fluid-filled lump that usually occurs in one breast, while a malignant tumor is a solid, abnormal growth that can usually be felt and is often accompanied by other concerning symptoms. Benign cysts are very common and usually don't need treatment, but it's still important to have them checked by a doctor to rule out anything more serious. If you're unsure whether a lump is a cyst or a tumor, it's always better to get it checked rather than try to self-diagnose.


# ***Testing of finetuned model on cpu***

In [4]:
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer

# # Replace with your actual Hugging Face repository
# model_id = "A-Asif/llama3.2-breastcancer-assistant"

# print("Loading tokenizer...")
# tokenizer = AutoTokenizer.from_pretrained(model_id)

# print("Loading model into CPU RAM (this might take a minute)...")
# # Loading in bfloat16 or float32 ensures broad CPU compatibility
# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     device_map="auto",
#     torch_dtype=torch.bfloat16,
# )

# # Craft your domain-specific medical prompt
# messages = [
#     {
#         "role": "system",
#         "content": "You are a warm, empathetic breast health assistant. You explain breast cancer topics in plain, supportive language. You never diagnose or give personalized medical advice, and you always recommend the person consult their doctor for decisions specific to their situation."
#         },
#     {
#         "role": "user",
#         "content": "Can you explain the difference between a benign breast cyst and a malignant tumor?"
#     }
# ]

# # Apply the Llama 3.2 chat template
# inputs = tokenizer.apply_chat_template(
#     messages,
#     tokenize=True,
#     add_generation_prompt=True,
#     return_tensors="pt"
# ) # Notice we do NOT use .to("cuda") here

# #print("\nGenerating response (CPU inference is slower than GPU, please wait)...\n")

# # Generate the response
# with torch.no_grad(): # Speeds up inference by disabling gradient tracking
#     outputs = model.generate(
#         input_ids=inputs,
#         max_new_tokens=300,
#         use_cache=True,
#         temperature=0.3,
#     )

# # Decode and print the output
# response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
# print("--- ASSISTANT RESPONSE ---")
# print(response)